# Academic Advisor – Model Training Notebook
Builds the machine learning model behind the **Academic Advisor** web app.

**Pipeline:** create training data → explore → train a Random Forest → evaluate → save `model.pkl`.

Run on Google Colab: *Runtime → Run all*. The last cell downloads `model.pkl`; put it in the `model/` folder of the project.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

## 2. Training data
The dataset is generated inside this notebook (600 synthetic student records, fixed random seed so it is reproducible) and saved as `training_data.csv`.

| Feature | Meaning |
|---|---|
| `cgpa` | Cumulative GPA on a 5.0 scale |
| `study_hours` | Self-study hours per week |
| `attendance` | Class attendance (%) |
| `carryovers` | Number of outstanding (failed) courses |
| `status` | **Target:** At Risk / Needs Improvement / On Track |

A weighted academic-performance score (plus random noise) decides the label. Replace this with real records (e.g. your own results) for a real-world advisor.

In [ ]:
np.random.seed(42)
n = 600

df = pd.DataFrame({
    "cgpa": np.random.uniform(1.0, 5.0, n).round(2),
    "study_hours": np.random.randint(0, 41, n),
    "attendance": np.random.randint(30, 101, n),
    "carryovers": np.random.randint(0, 7, n),
})

score = (df.cgpa * 20 + df.study_hours * 0.8 + df.attendance * 0.3
         - df.carryovers * 6 + np.random.normal(0, 4, n))

df["status"] = pd.cut(score, bins=[-1000, 70, 88, 1000],
                      labels=["At Risk", "Needs Improvement", "On Track"]).astype(str)

df.to_csv("training_data.csv", index=False)
df.head(10)

## 3. Explore the data

In [ ]:
print(df.shape)
print(df.status.value_counts())
df.describe()

In [ ]:
df.status.value_counts().plot(kind="bar", color=["#d9534f", "#f0ad4e", "#5cb85c"])
plt.title("Class distribution"); plt.ylabel("Students"); plt.xticks(rotation=0)
plt.show()

## 4. Train / test split

In [ ]:
X = df.drop("status", axis=1)
y = df["status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(len(X_train), "training rows,", len(X_test), "test rows")

## 5. Train the model (Random Forest)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

## 6. Evaluate

In [ ]:
pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print()
print(classification_report(y_test, pred))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred, labels=model.classes_),
                       display_labels=model.classes_).plot(cmap="Blues")
plt.show()

In [ ]:
pd.Series(model.feature_importances_, index=X.columns).sort_values().plot(kind="barh")
plt.title("Feature importance"); plt.show()

## 7. Try a sample student

In [ ]:
sample = pd.DataFrame([{"cgpa": 3.8, "study_hours": 20, "attendance": 85, "carryovers": 0}])
print(model.predict(sample)[0])

## 8. Save the model as a `.pkl` file

In [ ]:
joblib.dump(model, "model.pkl")
print("Saved model.pkl")

# On Colab this downloads the files; elsewhere it is skipped.
try:
    from google.colab import files
    files.download("model.pkl")
    files.download("training_data.csv")
except ImportError:
    pass